In [1]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Get a handle to the workspace. You can find the info on the workspace tab on ml.azure.com
ml_client = MLClient(
    credential=credential,
    subscription_id="972cf06a-de12-45b1-87f1-3724a2166e79",  # this will look like xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
    resource_group_name="Ads-Singularity-RG-01",
    workspace_name="ads-singularity-ws01-eastus",
)

# Verify that the handle works correctly.
# If you get an error here, modify your SUBSCRIPTION, RESOURCE_GROUP, and WS_NAME in the previous cell.
ws = ml_client.workspaces.get("ads-singularity-ws01-eastus")
print(ws.location, ":", ws.resource_group)

eastus : Ads-Singularity-RG-01


In [4]:
from azure.ai.ml import command, Input

# define the command
command_job = command(   
    experiment_name='hstu-retrieval',
    display_name="9803944_astrov6_pinsage_len32_event_type_test",
    code='.',
    inputs=dict(
        data=dict(
            type="uri_folder",
            path="azureml://datastores/bingads_algo_prod_networkprotection_c08/paths/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2.5_astrov6/event_seqs_11032025",
            mode="ro_mount",
        ),
        ads_semantic_embd=dict(
            type="uri_folder",
            path="azureml://datastores/bingads_algo_prod_networkprotection_c08/paths/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2.5_astrov6/vocabs_11032025/pinsage_len32_ads",
            mode="download",
        ),
        web_browsing_semantic_embd=dict(
            type="uri_folder",
            path="azureml://datastores/bingads_algo_prod_networkprotection_c08/paths/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2.5_astrov6/vocabs_11032025/pinsage_len32_web",
            mode="download",
        ),
        shopping_semantic_embd=dict(
            type="uri_folder",
            path="azureml://datastores/bingads_algo_prod_networkprotection_c08/paths/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2.5_astrov6/vocabs_11032025/pinsage_len32_shopping",
            mode="download",
        ),  
        ads_pure_corpus_embd=dict(
            type="uri_folder",
            path="azureml://datastores/bingads_algo_prod_networkprotection_c08/paths/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2.5_astrov6/vocabs_11032025/pinsage_len32_ads_pure_corpus_ta_pa_200m",
            mode="download",         
        ),
    ),
    outputs=dict(
        out_dir=dict(
            type="uri_folder",
            path="azureml://datastores/bingads_algo_prod_networkprotection_c08/paths/shares/bingads.hm/local/NativeAds/Relevance/Data/sequential/hstu/v2.5_astrov6/experiments",
            mode="rw_mount",
        ),
    ),
    # command="sleep infinity",
    command="python3 main.py --data_path ${{inputs.data}} --ads_semantic_embd_path ${{inputs.ads_semantic_embd}} --web_browsing_semantic_embd_path ${{inputs.web_browsing_semantic_embd}} --shopping_semantic_embd_path ${{inputs.shopping_semantic_embd}} --ads_pure_corpus_embd_path ${{inputs.ads_pure_corpus_embd}} --output_path ${{outputs.out_dir}} --gin_config_file=configs/ads/training_data_11032025_astrov6_pinsage_event_type.gin",
    environment="hstu:3.3_dev2",
    environment_variables={
        '_AZUREML_SINGULARITY_JOB_UAI':'/subscriptions/972cf06a-de12-45b1-87f1-3724a2166e79/resourceGroups/Ads-Singularity-RG-01/providers/Microsoft.ManagedIdentity/userAssignedIdentities/UAMI-ads-singularity-ws01-eastus',
        'DEFAULT_IDENTITY_CLIENT_ID':'bf17d0e1-473c-4dea-b246-e7345a2d8c25'
    },
    compute="/subscriptions/972cf06a-de12-45b1-87f1-3724a2166e79/resourceGroups/Ads-Singularity-RG-01/providers/Microsoft.MachineLearningServices/virtualclusters/ads-pme-H100",
    resources=dict(
        instance_count=1,
        instance_type="Singularity.ND96r_H100_v5",
        properties=dict(
            singularity=dict(
                interactive="false",
                imageVersion='',
                slaTier="Premium",
                priority="High",
                tensorboardLogDirectory="",
                enableAzmlInt="false"
            )
        )

    ),   
    distribution=dict(
        type="PyTorch",
        process_count_per_instance=8
    )
)

# submit the command
returned_job = ml_client.jobs.create_or_update(command_job)
# get a URL for the status of the job
returned_job.studio_url

Uploading hstu_retrieval (0.65 MBs): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 650439/650439 [00:00<00:00, 1058544.42it/s]


'https://ml.azure.com/runs/cyan_lion_y6hc9w649z?wsid=/subscriptions/972cf06a-de12-45b1-87f1-3724a2166e79/resourcegroups/Ads-Singularity-RG-01/workspaces/ads-singularity-ws01-eastus&tid=975f013f-7f24-47e8-a7d3-abc4752bf346'